# C2.6 · Benchmarks, reproducibility and the research harness

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *AI for Security*

Builds on **[C2.5 · Supply-chain research](https://spbreed.github.io/cyber-commons/lessons/C2.5.html)**.

| | |
|---|---|
| Tools used | Inspect, Cyber Commons eval harness, Llama 3.3, GLM-4.6, Kimi K2, Claude Opus 5 |

## What this lesson is

**What it covers.** Run one harness across three model families, separate the two effects, then contamination-check a public benchmark against a training window.

**Why a security engineer needs it.** Model effects and harness effects confounded, and published benchmarks overstating real-world capability. The control it builds is: multi-backbone runs on fixed seeds and corpora, plus contamination and construct-validity checks before any number is trusted.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Change the model and the harness at the same time and you have learned nothing about either. Separating those two effects is the whole job — and it is also how you read somebody else's published number without being misled by it.

> **At CyberTravels.** Change CyberTravels' model and its review harness in the same week and CyberTravels has learned nothing about either.

## 2 · The framework

```
   two effects, one number
   +-------------+     +--------------+
   |   model     |  x  |   harness    |  = the result you published
   +-------------+     +--------------+

   change one at a time, on fixed seeds and a fixed corpus.

   then the three checks on anybody's benchmark
   class balance -> the floor . held-out key -> a result . matcher -> real
```

The difference between a person who finds things and a capability that keeps
finding them is a harness: a suite, a target adapter, and recorded rates that
are comparable across runs.

Three properties make it a harness rather than a script:

1. **The suite is data, not code.** Adding a case must not require editing the
   runner.
2. **The target is an adapter.** Pointing it at a new build, a new model or a
   competitor's product should be one function.
3. **Results are comparable.** Same seed, same n, same scoring — so a delta
   means something.

The failure mode to avoid is a harness that only ever produces a number going
down, because the suite is only ever extended with cases the current build
already passes.

The same three properties are what let you **critique somebody else's
benchmark**, which is the other half of this job. Three questions decide whether
a published security number means anything, and all three are answerable from
the benchmark's own data:

1. **What is the class balance?** If one class dominates, a constant answer
   scores well. Report lift over the majority baseline, never the raw number.
2. **Is the key held out?** If the harness has seen the answers — through
   training, through prompt examples, through its own logs — the number is a
   training metric.
3. **How are files matched?** Bare-basename matching on a corpus that reuses
   filenames turns accuracy into a partly random variable.

## 3 · Where it breaks — a suite that only ever grows easier

The metric that makes a research programme look productive while measuring nothing: add cases the current build already passes, and the aggregate ASR falls every quarter.

## 4 · The same discipline, pointed at somebody else's benchmark

Dilution is one way a number lies. Three more are structural, and all three are checkable from the benchmark's own key: class balance, whether the key was held out, and how answers are matched to files.

## 5 · The procedure, as a skill

A control should move the surface it addresses and leave the others alone. The skill checks both halves with intervals, then dilutes the suite with cases everything blocks and watches the aggregate improve while nothing changed.

### The skill — [`skills/research/eval-suite-health-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/eval-suite-health-check/SKILL.md)

```yaml
name: eval-suite-health-check
description: >-
  Run an evaluation suite with confidence intervals, confirm a control moves one
  surface and not the others, and detect the suite being diluted by cases
  everything blocks. Use when an eval score improves and you need to know
  whether the system did.
allowed-tools: Read, Grep, Glob
```

# A suite that gets easier reports that you got better

An evaluation suite is an instrument, and instruments drift. Two properties keep
it honest: a control should move the surface it addresses and leave the others
alone, and the aggregate should not improve because somebody added cases
everything already blocks.

## When to use this

Whenever an eval number moves, before adding cases to a suite, and at any
regular review of a safety benchmark you rely on.

## Procedure

**1 — Run the baseline with intervals.** Per case and per surface. A point
estimate cannot support the comparison you are about to make.

**2 — Apply one control and re-run.** The prediction is specific: the surface it
addresses drops, with non-overlapping intervals, and the other surfaces do not
move. Both halves are the test — a control that moves everything is measuring
something other than the control.

**3 — Report per surface, never only in aggregate.** The aggregate hides both a
control that works and a control that broke something else.

**4 — Dilute the suite deliberately.** Add cases the target trivially blocks and
re-compute. Watch the aggregate improve while nothing about the system changed.
That demonstration is what justifies the next step.

**5 — Add a suite-health check.** Difficulty distribution, share of cases no
target has ever failed, and the date each case was added. A suite with a growing
share of trivial cases is reporting improvement it has not earned.

## Example

**Input** — the fixture committed at the top of [`scripts/eval_suite_health_check.py`](scripts/eval_suite_health_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
case    surface         rate              ci95
------------------------------------------------
INJ-01  injection      0.055    (0.033, 0.077)
INJ-02  injection      0.355    (0.308, 0.402)
INJ-03  injection      0.657    (0.611, 0.704)
INJ-04  injection      0.725    (0.681, 0.769)
IDN-01  identity       0.000        (0.0, 0.0)
IDN-02  identity       0.953    (0.932, 0.973)
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "baseline": [{"case": "str", "surface": "str", "rate": 0.0, "interval": [0.0, 0.0]}],
  "with_control": [{"surface": "str", "rate": 0.0, "interval": [0.0, 0.0], "moved": true}],
  "expected_unchanged": ["str"],
  "dilution": {"added_trivial": 0, "aggregate_before": 0.0, "aggregate_after": 0.0},
  "health": {"trivial_share": 0.0, "never_failed": 0, "oldest_case": "str"}
}
```

## Failure modes

- **Aggregate-only reporting.** It hides the two things you are looking for.
- **A control that moves every surface.** Investigate before celebrating.
- **Adding cases without recording difficulty.** The suite drifts easier and the
  score drifts up.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/eval-suite-health-check/scripts/eval_suite_health_check.py
SCRIPT = "skills/research/eval-suite-health-check/scripts/eval_suite_health_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The baseline suite reports per-case rates with intervals. Provenance reduces every injection case to about 0.02 with non-overlapping intervals, while identity and containment are unchanged. Adding 12 trivially-blocked cases cuts aggregate ASR by roughly 60% with no change to the build, and the suite-health check flags that suite as diluted. On the critique side: a skewed key gives a 0.875 floor before anyone answers anything, a leaked key scores a perfect 1.000, and answers naming the wrong directory score 1.000 under basename matching against 0.250 under path matching.

## Your turn

Check your own security regression suite for dilution — what fraction of its cases have ever failed? Under 30% and the aggregate number is mostly measuring how many easy cases you added. Then take the last benchmark someone quoted at you and find its majority baseline. Most published numbers are never reported against one.

---

**Next → [C2.7 · From finding to control, and to institutional capital](https://spbreed.github.io/cyber-commons/lessons/C2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*